# Fase 2 — Adquisición y EDA (Semana 3-4)

Referencia: `DOCUMENTOS ELABORADOS DE LA TESIS/Protocolo de Investigacion v1.md` (secciones 5 y 12).

**Alcance de este notebook:**
1. Descargar el dataset UCI id=697 y verificar su integridad contra la ficha técnica del protocolo.
2. Ejecutar el EDA sobre las 3 clases originales (Dropout / Enrolled / Graduate) — sin excluir, filtrar, normalizar ni particionar todavía.

**Fuera de alcance (no tocar aquí):** exclusión de "Enrolled", binarización, partición train/val/test, normalización, SMOTE. Eso es Fase 2 tardía (semana 4) y Fase 3 (semanas 5-6).

## 0. Entorno

Ejecutar en local (CPU, sin GPU necesaria — ver §10 del protocolo). Dependencias mínimas para esta fase: `ucimlrepo`, `pandas`, `numpy`, `matplotlib`, `seaborn`.

In [ ]:
# Ejecutar una sola vez si falta alguna dependencia
# %pip install ucimlrepo pandas numpy matplotlib seaborn

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from ucimlrepo import fetch_ucirepo

pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")

## 1. Adquisición del dataset (UCI id=697)

Fuente oficial: UCI Machine Learning Repository, DOI 10.24432/C5MC89 (Realinho et al., 2022).
Kaggle es solo espejo de descarga — no se cita como fuente.

In [ ]:
dataset = fetch_ucirepo(id=697)

X = dataset.data.features
y = dataset.data.targets

df = pd.concat([X, y], axis=1)
print(dataset.metadata.name)
print(dataset.metadata.doi)
df.head()

### 1.1 Verificación de integridad

Indicador de cierre (cronograma, semana 3): **0 discrepancias entre metadatos UCI y archivo local.**

Contra la ficha técnica del protocolo:
- Registros esperados: **4,424**
- Variables predictoras esperadas: **35**
- Clases esperadas: **Dropout, Enrolled, Graduate**

In [ ]:
n_registros = df.shape[0]
n_variables = X.shape[1]
clases = sorted(y.iloc[:, 0].unique().tolist())

print(f"Registros: {n_registros} (esperado: 4424) -> {'OK' if n_registros == 4424 else 'DISCREPANCIA'}")
print(f"Variables predictoras: {n_variables} (esperado: 35) -> {'OK' if n_variables == 35 else 'DISCREPANCIA'}")
print(f"Clases encontradas: {clases}")

In [ ]:
# Guardar copia local cruda, sin modificar, para trazabilidad (data/raw queda fuera de git, ver .gitignore)
ruta_salida = "../../data/raw/dataset_crudo_uci697.csv"
df.to_csv(ruta_salida, index=False)
print(f"Guardado: {ruta_salida}")

## 2. EDA — Distribución de clases

Sobre las 3 clases originales, **antes** de excluir "Enrolled". Debe aproximarse a lo ya documentado en el protocolo: Dropout 32.1%, Graduate 49.9%, Enrolled 17.9%.

In [ ]:
target_col = y.columns[0]

conteo = df[target_col].value_counts()
porcentaje = df[target_col].value_counts(normalize=True) * 100

resumen_clases = pd.DataFrame({"frecuencia": conteo, "porcentaje": porcentaje.round(2)})
resumen_clases

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
sns.countplot(data=df, x=target_col, order=conteo.index, ax=ax)
ax.set_title("Distribución de clases originales")
for i, v in enumerate(conteo.values):
    ax.text(i, v + 20, str(v), ha="center")
plt.show()

## 3. EDA — Valores faltantes

In [ ]:
faltantes = df.isna().sum()
faltantes = faltantes[faltantes > 0].sort_values(ascending=False)

if faltantes.empty:
    print("Sin valores faltantes detectados.")
else:
    display(faltantes)

## 4. EDA — Outliers (variables numéricas)

In [ ]:
num_cols = X.select_dtypes(include=[np.number]).columns.tolist()

n_cols_plot = 4
n_rows_plot = int(np.ceil(len(num_cols) / n_cols_plot))
fig, axes = plt.subplots(n_rows_plot, n_cols_plot, figsize=(4 * n_cols_plot, 3 * n_rows_plot))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    sns.boxplot(x=df[col], ax=axes[i])
    axes[i].set_title(col, fontsize=8)

for j in range(len(num_cols), len(axes)):
    axes[j].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Conteo de outliers por IQR, solo para documentar -- no se elimina nada en esta fase
def contar_outliers_iqr(serie):
    q1, q3 = serie.quantile([0.25, 0.75])
    iqr = q3 - q1
    limite_inf, limite_sup = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return ((serie < limite_inf) | (serie > limite_sup)).sum()

outliers_resumen = pd.Series({col: contar_outliers_iqr(df[col]) for col in num_cols}).sort_values(ascending=False)
outliers_resumen.to_frame("n_outliers_iqr")

## 5. EDA — Correlaciones entre variables predictoras

In [ ]:
corr = X[num_cols].corr()

fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(corr, cmap="coolwarm", center=0, ax=ax)
ax.set_title("Matriz de correlación — variables numéricas")
plt.show()

In [ ]:
# Pares con correlación alta (|r| > 0.8), candidatos a redundancia -- documentar, no eliminar aún
corr_pairs = corr.abs().unstack()
corr_pairs = corr_pairs[corr_pairs < 1.0].sort_values(ascending=False)
corr_pairs.drop_duplicates().head(15)

## 6. Resumen para el Informe EDA (entregable de la semana 3-4)

Completar con lo observado antes de pasar a la depuración (semana 4):

- Distribución de clases: `[ completar ]`
- ¿Coincide con lo documentado en el protocolo (32.1% / 49.9% / 17.9%)? `[ Sí / No, diferencia: ___ ]`
- Valores faltantes relevantes: `[ completar ]`
- Variables con más outliers: `[ completar ]`
- Pares de variables altamente correlacionadas (candidatas a redundancia): `[ completar ]`
- Observaciones que afectan el preprocesamiento de la Fase 3: `[ completar ]`